In [52]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.express as px
from pathlib import Path
from transformers import AutoTokenizer
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer


In [53]:
df = pd.read_csv("../data/ReadabilityRomanian - Sheet1.csv")
print(df.shape[0])
print(df.shape[1])
print(df.columns.tolist())

642
6
['ID', 'Author', 'Title', 'Pub Year', 'Categ', 'Excerpt']


In [54]:
display(df[["Categ", "Excerpt"]].head())

,Categ,Excerpt
0,Romane,"În femeile din romanele rusești găsesc, alătur..."
1,Nuvele,"Neputând să-l mai urmez, mă hotărâi să-l aștep..."
2,Nuvele,"— Al dracului ungurean, zicea locotenentul Fru..."
3,Basme,Se miră împăratul când auzi de la porc așa vor...
4,Basme,"Atunci Ioviță Făt-frumos încordă arcul, își lu..."


In [55]:
print(df.isna().sum())
print(df["ID"].duplicated().sum())
print(df["Excerpt"].duplicated().sum())

ID           0
Author      47
Title        0
Pub Year     0
Categ        0
Excerpt      1
dtype: int64
0
0


In [56]:
display(df[df["Excerpt"].isna()])
display(df[df["Author"].isna()].head())

,ID,Author,Title,Pub Year,Categ,Excerpt
30,429122,NaN,Închisă printre nebuni o săptămână,1934,Reportaje,NaN


,ID,Author,Title,Pub Year,Categ,Excerpt
17,295901,NaN,Cu paloșul,1900,Romane,"Cnezatul lui cuprindea satele Comănești, Băieș..."
30,429122,NaN,Închisă printre nebuni o săptămână,1934,Reportaje,NaN
34,56212,NaN,Amintiri din copilărie și adolescență,1900,Amintiri,Tatăl meu m-a iubit foarte mult. Avea obiceiul...
79,56211,NaN,Amintiri din copilărie și adolescență,1900,Amintiri,Sărăcia se ținea de noi. Dar eu n-aveam cum s-...
98,266730,NaN,Troița,1900,Nuvele,Și Cristos s-a mărit cît întreaga troiță; cu p...


In [57]:
df_clean = df.dropna(subset=["Excerpt"]).copy()
print(df_clean["Categ"].value_counts())

Categ
Basme        236
Nuvele       175
Romane       121
Schițe        38
Amintiri      36
Memorii       12
Eseuri         4
Epopei         4
Biografii      4
Povestiri      4
Jurnale        4
Reportaje      3
Name: count, dtype: int64


In [58]:
df_clean["word_count"] = df_clean["Excerpt"].str.split().str.len()
print(df_clean["word_count"].describe())

count    641.000000
mean      64.351014
std       29.232046
min       30.000000
25%       42.000000
50%       56.000000
75%       82.000000
max      149.000000
Name: word_count, dtype: float64


In [59]:
df_clean.groupby("Categ")["word_count"].agg(["count", "mean", "median"])

,count,mean,median
Categ,,,
Amintiri,36,74.055556,70.5
Basme,236,60.343220,52.5
Biografii,4,59.250000,58.5
Epopei,4,34.000000,31.0
Eseuri,4,52.750000,51.5
Jurnale,4,44.250000,44.0
Memorii,12,76.916667,70.5
Nuvele,175,67.674286,58.0
Povestiri,4,98.000000,110.5


In [60]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [61]:
lengths = [len(model.tokenizer.encode(t)) for t in df_clean["Excerpt"]]
print(max(lengths))
print(sum(n > model.max_seq_length for n in lengths))
texts = ["passage: " + t for t in df_clean["Excerpt"]]
embeddings = model.encode(texts, normalize_embeddings=True)

313
0


In [62]:
# results = []
# for size in [5, 10, 15, 20]:
#     for samples in [2, 5, 10]:
#         labels = HDBSCAN(min_cluster_size=size,min_samples=samples).fit_predict(reduced)
#         results.append({"min_cluster_size": size,"min_samples": samples,"clusters": len(set(labels) - {-1}),"noise_%": round(100 * (labels ==-1).mean(), 1)})
# pd.DataFrame(results)

In [63]:
results = []
for k in range(2, 11):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(embeddings)
    results.append((k, silhouette_score(embeddings, labels, metric="cosine")))
pd.DataFrame(results, columns=["k", "silhouette"])

,k,silhouette
0,2,0.055633
1,3,0.039320
2,4,0.038890
3,5,0.022807
4,6,0.023997
5,7,0.017394
6,8,0.018845
7,9,0.018778
8,10,0.017327


In [64]:
reduced_10 = UMAP(n_components=10,n_neighbors=15,metric="cosine",random_state=42).fit_transform(embeddings)
clusterer = HDBSCAN(min_cluster_size=15, min_samples=2)
labels = clusterer.fit_predict(reduced_10)
reduced_2 = UMAP(n_components=2,n_neighbors=15,metric="cosine",random_state=42).fit_transform(embeddings)

c:\readability & eeg\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\readability & eeg\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [65]:
df_vis = df_clean.copy()
df_vis["x"] = reduced_2[:, 0]
df_vis["y"] = reduced_2[:, 1]
df_vis["cluster"] = labels
df_vis["cluster_label"] = df_vis["cluster"].apply(
    lambda x: "Noise" if x == -1 else f"Cluster {x}"
)
import textwrap
df_vis["excerpt_hover"] = df_vis["Excerpt"].apply(
    lambda t: "<br>".join(textwrap.wrap(t, width=65))
)
fig = px.scatter(
    df_vis,
    x="x",
    y="y",
    color="cluster_label",
    hover_name="Title",
    custom_data=["excerpt_hover", "cluster", "Categ", "Pub Year", "Author"],
    title="Exploratory Clustering E5"
)
fig.update_traces(
    marker=dict(size=8),
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "Cluster: %{customdata[1]}<br>"
        "Category: %{customdata[2]}<br>"
        "Year: %{customdata[3]}<br>"
        "Author: %{customdata[4]}<br><br>"
        "%{customdata[0]}"
        "<extra></extra>"
    )
)
fig.update_layout(
    height=850,
    xaxis_visible=False,
    yaxis_visible=False,
    legend_title_text="Cluster",
    hoverlabel=dict(align="left", font_size=12)
)
Path("../docs").mkdir(exist_ok=True)
fig.write_html(
    "../docs/E5.html",
    include_plotlyjs=True,
)

In [66]:
tokenizer_ro = AutoTokenizer.from_pretrained("dumitrescustefan/bert-base-romanian-cased-v1")
lengths_ro = [len(tokenizer_ro.encode(t)) for t in df_clean["Excerpt"]]
max(lengths_ro), sum(n > 512 for n in lengths_ro)

(245, 0)

In [67]:
tokenizer_ro = AutoTokenizer.from_pretrained(
    "dumitrescustefan/bert-base-romanian-cased-v1"
)

In [68]:
model_ro = AutoModel.from_pretrained("dumitrescustefan/bert-base-romanian-cased-v1").to("cuda").eval()
texts_ro = df_clean["Excerpt"].tolist()
embeddings_ro = []
with torch.no_grad():
    for i in range(0, len(texts_ro), 16):
        batch = tokenizer_ro(
            texts_ro[i:i+16],
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to("cuda")
        output = model_ro(**batch).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1)
        pooled = (output * mask).sum(dim=1) / mask.sum(dim=1)
        embeddings_ro.append(pooled.cpu())
embeddings_ro = F.normalize(torch.cat(embeddings_ro), dim=1).numpy()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dumitrescustefan/bert-base-romanian-cased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [69]:
reduced_ro = UMAP(
    n_components=10,
    n_neighbors=15,
    metric="cosine",
    random_state=42
).fit_transform(embeddings_ro)
labels_ro = HDBSCAN(
    min_cluster_size=20,
    min_samples=2
).fit_predict(reduced_ro)
pd.Series(labels_ro).value_counts()

c:\readability & eeg\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


 0    549
 1     88
-1      4
Name: count, dtype: int64

In [70]:
results_ro = []
for size in [5, 10, 15, 20, 30, 40]:
    for samples in [1, 2, 5, 10]:
        labels = HDBSCAN(
            min_cluster_size=size,
            min_samples=samples
        ).fit_predict(reduced_ro)
        results_ro.append({
            "min_cluster_size": size,
            "min_samples": samples,
            "clusters": len(set(labels) - {-1}),
            "noise_%": round(100 * (labels == -1).mean(), 1),
            "largest_cluster": max(
                (labels == c).sum() for c in set(labels) if c != -1
            ) if len(set(labels) - {-1}) else 0
        })
pd.DataFrame(results_ro)

,min_cluster_size,min_samples,clusters,noise_%,largest_cluster
0,5,1,42,27.1,34
1,5,2,36,32.1,33
2,5,5,2,2.8,536
3,5,10,2,4.4,526
4,10,1,2,0.3,549
5,10,2,2,0.6,549
6,10,5,2,2.8,536
7,10,10,2,4.4,526
8,15,1,2,0.3,549
9,15,2,2,0.6,549


In [71]:
labels_ro = HDBSCAN(
    min_cluster_size=5,
    min_samples=2
).fit_predict(reduced_ro)
reduced_2_ro = UMAP(n_components=2,n_neighbors=15,metric="cosine",random_state=42).fit_transform(embeddings_ro)
df_vis_ro = df_clean.copy()
df_vis_ro["x"] = reduced_2_ro[:, 0]
df_vis_ro["y"] = reduced_2_ro[:, 1]
df_vis_ro["cluster"] = labels_ro
df_vis_ro["cluster_label"] = df_vis_ro["cluster"].apply(lambda x: "Noise" if x == -1 else f"Cluster {x}")
df_vis_ro["excerpt_hover"] = df_vis_ro["Excerpt"].apply(lambda t: "<br>".join(textwrap.wrap(t, width=65)))
fig_ro = px.scatter(
    df_vis_ro,
    x="x",
    y="y",
    color="cluster_label",
    hover_name="Title",
    custom_data=["excerpt_hover", "cluster", "Categ", "Pub Year", "Author"],
    title="Exploratory Clustering Romanian BERT"
)
fig_ro.update_traces(
    marker=dict(size=8),
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "Cluster: %{customdata[1]}<br>"
        "Category: %{customdata[2]}<br>"
        "Year: %{customdata[3]}<br>"
        "Author: %{customdata[4]}<br><br>"
        "%{customdata[0]}"
        "<extra></extra>"
    )
)
fig_ro.update_layout(
    height=850,
    xaxis_visible=False,
    yaxis_visible=False,
    legend_title_text="Cluster",
    hoverlabel=dict(align="left", font_size=12)
)
Path("../docs").mkdir(exist_ok=True)

fig_ro.write_html(
    "../docs/romanian_bert.html",
    include_plotlyjs=True
)

c:\readability & eeg\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
